In [18]:
def init_tokenizer():
    with open('vocab_chars.txt', 'w') as f:
        for i in range(256):
            f.write(f"{i}: {repr(chr(i))}\n")
    
    print("vocab_chars.txt generado con 256 tokens ASCII (0-255)")

In [21]:
#encode the text
vocab = {}
with open('vocab_chars.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        token_id, char_repr = line.split(': ', 1)
        vocab[int(token_id)] = eval(char_repr)

print(len(vocab))
print(vocab)

256
{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '\x1e', 31: '\x1f', 32: ' ', 33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93: ']', 94: '^', 95: '_', 96: '`', 97: 'a', 98: 'b', 99: 'c', 100: 'd', 101:

In [29]:
#safe text into string
with open('frankenstein.txt', 'r') as f:
    text = f.read()

print(len(text))
print(text[:200])
print(type(text))

437389
Letter 1
_To Mrs. Saville, England._
St. Petersburgh, Dec. 11th, 17—.
You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
<class 'str'>


In [ ]:
def encode(text, vocab):
    # invertir vocab: char/string -> id
    str_to_id = {v: k for k, v in vocab.items()}
    tokens = [str_to_id[ch] for ch in text]
    # aplicar merges: intentar reemplazar pares conocidos
    changed = True
    while changed:
        changed = False
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1:
                pair_str = vocab.get(tokens[i], '') + vocab.get(tokens[i+1], '')
                if pair_str in str_to_id and str_to_id[pair_str] != tokens[i]:
                    new_tokens.append(str_to_id[pair_str])
                    i += 2
                    changed = True
                    continue
            new_tokens.append(tokens[i])
            i += 1
        tokens = new_tokens
    return tokens

def decode(tokens, vocab):
    return ''.join(vocab[t] for t in tokens)

In [ ]:
def tokenizer_train(text, vocab, num_merges=20):
    # tokenizar texto inicial: cada caracter -> su id
    str_to_id = {v: k for k, v in vocab.items()}
    tokens = [str_to_id[ch] for ch in text]

    for i in range(num_merges):
        # contar parejas adyacentes
        pairs = {}
        for j in range(len(tokens) - 1):
            pair = (tokens[j], tokens[j+1])
            pairs[pair] = pairs.get(pair, 0) + 1

        # encontrar la pareja más frecuente
        best_pair = max(pairs, key=pairs.get)
        best_count = pairs[best_pair]

        # crear nuevo token
        new_id = len(vocab)
        new_str = vocab[best_pair[0]] + vocab[best_pair[1]]
        vocab[new_id] = new_str

        # reemplazar todas las ocurrencias de la pareja en tokens
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and tokens[j] == best_pair[0] and tokens[j+1] == best_pair[1]:
                new_tokens.append(new_id)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        tokens = new_tokens

        print(f"Merge {i+1}: {best_pair} -> {new_id} ('{new_str}') | count: {best_count} | tokens: {len(tokens)}")

    return vocab, tokens